# 05 - End-to-End Orchestration

This notebook demonstrates the complete workflow: loading invoices, extracting data, validating, and submitting to Fakturoid.

In [1]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor, InvoiceData
from src.fakturoid_client import FakturoidClient
from src.config import config
from src.agent import InvoiceProcessingAgent
import json
from pathlib import Path
from datetime import datetime

In [2]:
# Initialize agent
agent = InvoiceProcessingAgent(config)

print("Invoice Processing Agent initialized")
print(f"Mode: {config.processing.mode}")
print(f"Auto-submit: {config.processing.auto_submit}")
print(f"Invoices directory: {config.directories.invoices}")

Invoice Processing Agent initialized
Mode: manual  # Options: auto, manual, both
Auto-submit: False
Invoices directory: /Users/pavelzverina/AiProjects/fakturoid/data/invoices


In [3]:
# Test connections
print("Testing connections...")
if agent.test_connections():
    print("✓ All connections successful!")
else:
    print("✗ Connection test failed. Check your credentials.")

2025-10-08 21:35:24,283 - src.agent - INFO - Testing Fakturoid connection...


Testing connections...


2025-10-08 21:35:26,086 - src.agent - INFO - All connections successful


✓ All connections successful!


In [ ]:
# Display AI usage summary for the batch
agent.ai_extractor.print_usage_summary()


In [4]:
# Process all invoices with manual review
print("="*60)
print("PROCESSING ALL INVOICES")
print("="*60)

# Get list of files first
files = agent.doc_processor.list_invoice_files()
print(f"\nFound {len(files)} invoice files")

if len(files) == 0:
    print(f"\nNo invoice files found in: {agent.invoices_dir}")
    print("Please add PDF or image files to process.")
else:
    # Process batch with review enabled
    results = agent.process_batch(review=True, max_files=None)
    
    print(f"\n{'='*60}")
    print("PROCESSING SUMMARY")
    print(f"{'='*60}")
    submitted = sum(1 for r in results if r['status'] == 'submitted')
    extracted = sum(1 for r in results if r['status'] == 'extracted')
    errors = sum(1 for r in results if r['status'] == 'error')
    
    print(f"Total invoices: {len(results)}")
    print(f"Submitted: {submitted}")
    print(f"Extracted (pending review): {extracted}")
    print(f"Failed: {errors}")

2025-10-08 21:35:31,687 - src.agent - INFO - Processing 4 invoice files
2025-10-08 21:35:31,687 - src.agent - INFO - Processing file: Alien Isolation.pdf


PROCESSING ALL INVOICES

Found 4 invoice files


2025-10-08 21:35:36,836 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 21:35:36,846 - src.agent - INFO - Extracted data from Alien Isolation.pdf
2025-10-08 21:35:36,847 - src.agent - INFO - Processing file: OpenAI-Invoice-3D6B9186-0031.pdf



EXTRACTED INVOICE DATA
Invoice Number: 786940972572357
Issue Date: 2025-10-02
Supplier: Sony Interactive Entertainment Network Europe Limited
Total Amount: 207.25 CZK

Line Items:
  1. Alien: Isolation (Game) - 1 x 0



2025-10-08 21:35:41,345 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 21:35:41,347 - src.agent - INFO - Extracted data from OpenAI-Invoice-3D6B9186-0031.pdf
2025-10-08 21:35:41,348 - src.agent - INFO - Processing file: OrderSummary202509013059566415002039.png



EXTRACTED INVOICE DATA
Invoice Number: 3D6B9186-0031
Issue Date: 2025-10-06
Supplier: OpenAI, LLC
Total Amount: 20.0 USD
Due Date: 2025-10-06

Line Items:
  1. ChatGPT Plus Subscription Oct 6 – Nov 6, 2025 - 1 x 20.0



2025-10-08 21:35:46,771 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 21:35:46,772 - src.agent - INFO - Extracted data from OrderSummary202509013059566415002039.png
2025-10-08 21:35:46,772 - src.agent - INFO - Processing file: google-workspace-5369924648.pdf



EXTRACTED INVOICE DATA
Invoice Number: 3059566415002039
Issue Date: 2025-08-25
Supplier: Haiming One Store Store
Total Amount: 105.07 CZK

Line Items:
  1. Tuya WiFi Smart IR Remote Control - 1 x 105.07



2025-10-08 21:35:52,114 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 21:35:52,116 - src.agent - INFO - Extracted data from google-workspace-5369924648.pdf
2025-10-08 21:35:52,116 - src.agent - INFO - Batch processing complete: 0 submitted, 0 errors out of 4 total



EXTRACTED INVOICE DATA
Invoice Number: 5369924648
Issue Date: 2025-09-30
Supplier: Google Cloud EMEA Limited
Total Amount: 8.1 EUR

Line Items:
  1. Google Workspace Business Starter - 1 x 0


PROCESSING SUMMARY
Total invoices: 4
Submitted: 0
Extracted (pending review): 4
Failed: 0


In [5]:
# Show detailed results
if 'results' in locals() and results:
    print("\nDetailed Results:")
    for result in results:
        print(f"\n{'-'*60}")
        print(f"File: {result['file']}")
        print(f"Status: {result['status']}")
        
        if result['status'] == 'submitted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
            if result.get('fakturoid_response'):
                print(f"Fakturoid ID: {result['fakturoid_response'].get('id')}")
                print(f"Fakturoid Number: {result['fakturoid_response'].get('number')}")
                print(f"URL: {result['fakturoid_response'].get('html_url')}")
        
        elif result['status'] == 'extracted':
            if result.get('extracted_data'):
                data = result['extracted_data']
                print(f"Invoice Number: {data.get('invoice_number', 'N/A')}")
                print(f"Supplier: {data.get('supplier_name', 'N/A')}")
                print(f"Amount: {data.get('total_amount', 'N/A')} {data.get('currency', 'CZK')}")
                print("⚠️  Awaiting manual review/approval")
        
        elif result['status'] == 'error':
            print(f"Error: {result.get('error', 'Unknown error')}")
else:
    print("No results to display. Run the previous cell first.")


Detailed Results:

------------------------------------------------------------
File: Alien Isolation.pdf
Status: extracted
Invoice Number: 786940972572357
Supplier: Sony Interactive Entertainment Network Europe Limited
Amount: 207.25 CZK
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: OpenAI-Invoice-3D6B9186-0031.pdf
Status: extracted
Invoice Number: 3D6B9186-0031
Supplier: OpenAI, LLC
Amount: 20.0 USD
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: OrderSummary202509013059566415002039.png
Status: extracted
Invoice Number: 3059566415002039
Supplier: Haiming One Store Store
Amount: 105.07 CZK
⚠️  Awaiting manual review/approval

------------------------------------------------------------
File: google-workspace-5369924648.pdf
Status: extracted
Invoice Number: 5369924648
Supplier: Google Cloud EMEA Limited
Amount: 8.1 EUR
⚠️  Awaiting manual review/approval


In [6]:
# Process a single invoice with detailed steps
files = agent.doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"Processing single invoice: {test_file.name}")
    print("="*60)
    
    # Use the agent's process_file method
    print("\n📄 Processing invoice...")
    result = agent.process_file(test_file, review=True)
    
    print(f"\n{'='*60}")
    print(f"Status: {result['status']}")
    print(f"{'='*60}")
    
    if result['status'] == 'error':
        print(f"\n✗ Error: {result['error']}")
    
    elif result['status'] == 'extracted':
        print("\n✓ Invoice data extracted successfully!")
        print("⚠️  Review the data above. To submit, set auto_submit=True")
    
    elif result['status'] == 'submitted':
        print("\n✓ Invoice submitted to Fakturoid!")
        if result.get('fakturoid_response'):
            resp = result['fakturoid_response']
            print(f"   ID: {resp.get('id')}")
            print(f"   Number: {resp.get('number')}")
            print(f"   URL: {resp.get('html_url')}")
else:
    print("No invoice files found.")
    print(f"Please add PDF or image files to: {agent.invoices_dir}")

2025-10-08 21:36:42,766 - src.agent - INFO - Processing file: Alien Isolation.pdf


Processing single invoice: Alien Isolation.pdf

📄 Processing invoice...


2025-10-08 21:36:47,402 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-10-08 21:36:47,403 - src.agent - INFO - Extracted data from Alien Isolation.pdf



EXTRACTED INVOICE DATA
Invoice Number: 786940972572357
Issue Date: 2025-10-02
Supplier: Sony Interactive Entertainment Network Europe Limited
Total Amount: 207.25 CZK

Line Items:
  1. Alien: Isolation (Game) - 1 x 0


Status: extracted

✓ Invoice data extracted successfully!
⚠️  Review the data above. To submit, set auto_submit=True


In [7]:
# Manual workflow: Extract, review, then submit
# This demonstrates how to submit an already-extracted invoice

if 'result' in locals() and result.get('status') == 'extracted':
    print("Manual submission workflow")
    print("="*60)
    
    # Get the extracted data
    invoice_data = InvoiceData(**result['extracted_data'])
    file_path = agent.invoices_dir / result['file']
    
    print(f"\nReady to submit: {result['file']}")
    print(f"Supplier: {invoice_data.supplier_name}")
    print(f"Amount: {invoice_data.total_amount} {invoice_data.currency or 'CZK'}")
    
    # Uncomment the following lines to actually submit:
    submit_result = agent.submit_extracted(file_path, invoice_data)
    
    if submit_result['status'] == 'submitted':
        resp = submit_result['fakturoid_response']
        print(f"\n✓ Submitted to Fakturoid!")
        print(f"   Expense number: {resp.get('number')}")
        print(f"   ID: {resp.get('id')}")
        print(f"\n📁 File renamed and moved to processed/")
        print(f"   New name will be: {resp.get('number')} - {invoice_data.supplier_name} - ...")
    else:
        print(f"\n✗ Error: {submit_result.get('error')}")
    
    print("\n⚠️  Uncomment the code above to submit")
else:
    print("No extracted invoice available.")
    print("Run the previous cell first to extract invoice data.")

Manual submission workflow

Ready to submit: Alien Isolation.pdf
Supplier: Sony Interactive Entertainment Network Europe Limited
Amount: 207.25 CZK
⚙ Creating new subject: Sony Interactive Entertainment Network Europe Limited
  ✓ Subject created with ID: 28430207

🔍 DEBUG - Sending to Fakturoid API:
{
  "subject_id": 28430207,
  "lines": [
    {
      "name": "This is not a VAT/GST invoice. Email sent from a send-only address, do not reply.",
      "quantity": "1.0",
      "unit_price": "171.28",
      "vat_rate": 21
    }
  ],
  "original_number": "786940972572357",
  "document_type": "invoice",
  "issued_on": "2025-10-02",
  "taxable_fulfillment_due": "2025-10-02",
  "received_on": "2025-10-02",
  "description": "This is not a VAT/GST invoice. Email sent from a send-only address, do not reply.",
  "currency": "CZK"
}


2025-10-08 21:37:36,425 - src.agent - INFO - Submitted expense 786940972572357 to Fakturoid
2025-10-08 21:37:36,427 - src.agent - INFO - Moved Alien Isolation.pdf → FP20250181 - Sony Interactive Entertainment Network Europe Limited - Alien- Isolation (Game) - Alien Isolation.pdf



✓ Submitted to Fakturoid!
   Expense number: FP20250181
   ID: 3468126

📁 File renamed and moved to processed/
   New name will be: FP20250181 - Sony Interactive Entertainment Network Europe Limited - ...

⚠️  Uncomment the code above to submit


## 📁 Processed File Naming

When an invoice is successfully submitted, it's moved to the `processed/` directory with a descriptive name:

**Format:**
```
[Expense Number] - [Supplier] - [Description] - [Original Name].[ext]
```

**Example:**
```
FP20240189 - Alza.cz a.s. - Test invoice for API integration - faktura.pdf
```

**Components:**
1. **Expense Number** - From Fakturoid (e.g., `FP20240189`)
2. **Supplier** - Company name (sanitized)
3. **Description** - First line item description or notes (max 50 chars)
4. **Original Name** - Your original filename

This makes it easy to find and identify processed invoices!

In [9]:
# Check processed invoices directory
processed_dir = config.directories.processed
if processed_dir.exists():
    processed_files = [f for f in processed_dir.glob("*") if f.is_file() and not f.name.startswith('.')]
    
    print(f"Processed files directory: {processed_dir}")
    print(f"Number of processed files: {len(processed_files)}")
    
    if processed_files:
        print("\nProcessed files (most recent first):")
        sorted_files = sorted(processed_files, key=lambda x: x.stat().st_mtime, reverse=True)
        for f in sorted_files[:10]:
            mtime = datetime.fromtimestamp(f.stat().st_mtime)
            print(f"  - {f.name} ({mtime.strftime('%Y-%m-%d %H:%M')})") 
        if len(processed_files) > 10:
            print(f"  ... and {len(processed_files) - 10} more")
    else:
        print("\n📁 No processed files yet")
else:
    print(f"Processed directory doesn't exist yet: {processed_dir}")
    print("It will be created when the first invoice is submitted.")

Processed files directory: /Users/pavelzverina/AiProjects/fakturoid/data/processed
Number of processed files: 1

Processed files (most recent first):
  - FP20250181 - Sony Interactive - Alien Isolation.pdf (2025-10-02 13:43)
